# BreakHis EDA

Class balance, per-magnification counts, and sample grids for the BreakHis dataset.

In [ ]:
import sys
sys.path.append('..')

from src.data.dataset import BreakHisDataset

ds = BreakHisDataset('../data/BreaKHis_v1')
len(ds)

## Class balance

In [ ]:
import matplotlib.pyplot as plt

from src.data.eda import class_counts

counts = class_counts(ds.samples)
labels = ['benign', 'malignant']
values = [counts.get(0, 0), counts.get(1, 0)]

plt.bar(labels, values)
plt.ylabel('image count')
plt.title('Class balance')
plt.show()

## Image counts per magnification

In [ ]:
from src.data.eda import magnification_counts

mag_counts = magnification_counts(ds.samples)
mags = ['40', '100', '200', '400']

plt.bar(mags, [mag_counts.get(m, 0) for m in mags])
plt.xlabel('magnification')
plt.ylabel('image count')
plt.title('Images per magnification level')
plt.show()

## Per-subtype breakdown

In [ ]:
from src.data.eda import subtype_counts

sub_counts = subtype_counts(ds.samples)

plt.barh(list(sub_counts.keys()), list(sub_counts.values()))
plt.xlabel('image count')
plt.title('Images per tumor subtype')
plt.tight_layout()
plt.show()

## Sample grid

In [ ]:
import random

label_names = {0: 'benign', 1: 'malignant'}
sample_idxs = random.sample(range(len(ds)), min(9, len(ds)))  # NOSONAR: notebook display sampling only

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, idx in zip(axes.flat, sample_idxs):
    image, label = ds[idx]
    ax.imshow(image)
    ax.set_title(label_names[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

## Image size / quality checks

In [ ]:
from collections import Counter

sizes = Counter()
corrupt = []

for sample in ds.samples:
    try:
        from PIL import Image
        with Image.open(sample['path']) as img:
            sizes[img.size] += 1
    except Exception as e:
        corrupt.append((sample['path'], str(e)))

print('Image sizes found:', dict(sizes))
print(f'Corrupt/unreadable files: {len(corrupt)}')
for path, err in corrupt[:10]:
    print(' ', path, '-', err)

## Augmented samples

Visualize the training-time augmentation pipeline (flips, rotation, color
jitter) applied to a few images, to sanity-check it isn't distorting tissue
structure beyond recognition.

In [ ]:
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD, train_transform

augment = train_transform()
sample_idx = sample_idxs[0]
original_image, _ = ds[sample_idx]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(original_image)
axes[0].set_title('original')
axes[0].axis('off')

for ax in axes[1:]:
    augmented = augment(original_image)
    # unnormalize for display
    img = augmented.permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    ax.imshow(img.clip(0, 1))
    ax.set_title('augmented')
    ax.axis('off')

plt.tight_layout()
plt.show()